In [80]:
import os
import torch
import torch.nn as nn
from torch.optim import Adam

from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor
PATH = os.getcwd()
from tqdm import tqdm
from matplotlib.pyplot import imshow

In [81]:
mnist_train = FashionMNIST(root=PATH, download=True, train=True, transform=ToTensor()) # [feature, label]
mnist_test = FashionMNIST(root=PATH, download=True, train=False, transform=ToTensor()) # [feature, label]

class FeatureOnlyFMNIST(Dataset):
    def __init__(self, raw_dataset):
        self.raw = raw_dataset
        
    def __len__(self):
        return len(self.raw)
    
    def __getitem__(self, idx):
        img, _ = self.raw[idx]
        return img
    
data = FeatureOnlyFMNIST(mnist_train)

train_dataloader = DataLoader(data, batch_size=32, shuffle=True)

In [82]:
data[0].shape, data[0].std()

(torch.Size([1, 28, 28]), tensor(0.3994))

In [83]:
next(iter(train_dataloader)).shape

torch.Size([32, 1, 28, 28])

In [92]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 28 * 28),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon

In [98]:
vae = VAE()

optim = Adam(vae.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
EPOCHS = 16

dataloader = DataLoader(FeatureOnlyFMNIST(mnist_train), batch_size=32, shuffle=True)

for epoch in tqdm(range(EPOCHS)):
    total_loss = 0
    for batch in dataloader:    
        optim.zero_grad()

        pred = vae(batch)
        loss = loss_fn(pred, batch)
        loss.backward()
        optim.step()
        
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}, Loss: {total_loss / len(dataloader):.4f}")

  6%|███████████▏                                                                                                                                                                       | 1/16 [00:03<00:51,  3.43s/it]

Epoch 1, Loss: 0.0242


 12%|██████████████████████▍                                                                                                                                                            | 2/16 [00:06<00:45,  3.28s/it]

Epoch 2, Loss: 0.0147


 19%|█████████████████████████████████▌                                                                                                                                                 | 3/16 [00:09<00:42,  3.27s/it]

...

Epoch 14, Loss: 0.0094


 94%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉           | 15/16 [00:56<00:04,  4.02s/it]

Epoch 15, Loss: 0.0094


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [01:00<00:00,  3.79s/it]

Epoch 16, Loss: 0.0093
